# 03 — Pandas Analysis (10 questions, each with insight)

In [1]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect("data/tmdb.db")
movies       = pd.read_sql("SELECT * FROM movies", conn)
genres       = pd.read_sql("SELECT * FROM genres", conn)
movie_genres = pd.read_sql("SELECT * FROM movie_genres", conn)
cast         = pd.read_sql("SELECT * FROM cast", conn)
crew         = pd.read_sql("SELECT * FROM crew", conn)
keywords     = pd.read_sql("SELECT * FROM movie_keywords", conn)

movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
movies["release_year"] = movies["release_date"].dt.year

### Q1 — Unique counts across tables

In [2]:
unique_counts = {
    "movies":       movies.movie_id.nunique(),
    "genres":       genres.genre_id.nunique(),
    "cast_members": cast.person_id.nunique(),
    "crew_members": crew.person_id.nunique(),
}
pd.Series(unique_counts, name="unique_count")

movies           2503
genres             19
cast_members    11300
crew_members     2848
Name: unique_count, dtype: int64

### Q2 — Movies with 0 / missing budget or revenue

In [3]:
invalid_fin = movies[(movies.budget.fillna(0) == 0) | (movies.revenue.fillna(0) == 0)]
print(f"Movies with 0/missing budget or revenue: {len(invalid_fin)} of {len(movies)}")
invalid_fin[["movie_id", "title", "budget", "revenue"]].head(20)

Movies with 0/missing budget or revenue: 133 of 2503


,movie_id,title,budget,revenue
66,147,The 400 Blows,0.0,0.0
158,422,8½,0.0,0.0
253,653,Nosferatu,0.0,27964.0
314,797,Persona,0.0,250000.0
334,832,M,0.0,35274.0
422,1398,Stalker,120000.0,0.0
717,8274,American Pie Presents: Band Camp,7000000.0,0.0
730,8390,"Definitely, Maybe",0.0,55447968.0
768,8923,Green Street Hooligans,0.0,4333271.0
771,8965,Atlantis: Milo's Return,0.0,0.0


**Insight / Decision:** TMDB stores `0` to mean *unknown*, not *free*.
For all profit/ROI analysis we therefore **exclude** rows where
`budget <= 0` or `revenue <= 0`. Those rows are still kept for rating,
genre and popularity analyses.

### Q3 — Profit and ROI for movies with valid figures

In [4]:
valid = movies[(movies.budget > 0) & (movies.revenue > 0)].copy()
valid["profit"] = valid.revenue - valid.budget
valid["roi"]    = valid.profit / valid.budget
valid[["title", "budget", "revenue", "profit", "roi"]].sort_values("roi", ascending=False).head(10)

,title,budget,revenue,profit,roi
1169,Paranormal Activity,15000.0,193355800.0,193340800.0,12889.386667
562,The Blair Witch Project,60000.0,248639099.0,248579099.0,4142.984983
2497,Obsession,750000.0,474895195.0,474145195.0,632.193593
584,Bambi,858000.0,267447150.0,266589150.0,310.709965
872,Mad Max,350000.0,100000000.0,99650000.0,284.714286
945,Night of the Living Dead,114000.0,30236452.0,30122452.0,264.232035
1190,The Texas Chain Saw Massacre,140000.0,30922680.0,30782680.0,219.876286
375,Halloween,325000.0,70260597.0,69935597.0,215.186452
2191,Dragon Ball Super: Broly,1000000.0,125002821.0,124002821.0,124.002821
152,Snow White and the Seven Dwarfs,1488423.0,184925486.0,183437063.0,123.242561


**Insight:** A handful of low-budget films produce ROI multiples far above blockbusters —
small-budget genre bets are often more capital-efficient than big-budget productions.

### Q4 — Top 15 by ROI (budget ≥ $1M)

In [5]:
top_roi = valid[valid.budget >= 1_000_000].sort_values("roi", ascending=False).head(15)
top_roi[["title", "budget", "revenue", "roi"]]

,title,budget,revenue,roi
2191,Dragon Ball Super: Broly,1000000.0,125002821.0,124.002821
152,Snow White and the Seven Dwarfs,1488423.0,184925486.0,123.242561
1211,The Rocky Horror Picture Show,1400000.0,171181400.0,121.272429
412,Rocky,1000000.0,117253345.0,116.253345
300,Gone with the Wind,4000000.0,402352579.0,99.588145
794,The Jungle Book,4000000.0,378000000.0,93.500000
1001,Cinderella,2900000.0,263600000.0,89.896552
78,Saw,1200000.0,104045735.0,85.704779
1055,One Hundred and One Dalmatians,3600000.0,303000000.0,83.166667
221,E.T. the Extra-Terrestrial,10500000.0,797307407.0,74.934039


**Insight:** Even after filtering to ≥ $1M budgets, ROI outliers still return 20×+ —
useful signal for studios seeking low-risk, high-return projects.

### Q5 — Movies released per year

In [6]:
by_year = movies.dropna(subset=["release_year"]).groupby("release_year").size().sort_index()
print("Year with most releases:", int(by_year.idxmax()), "→", int(by_year.max()), "movies")
by_year.tail(20)

Year with most releases: 2016 → 121 movies


release_year
2007     78
2008     81
2009     85
2010     90
2011     97
2012     82
2013    110
2014    105
2015     97
2016    121
2017    109
2018    102
2019    100
2020     52
2021     67
2022     65
2023     62
2024     43
2025     26
2026      8
dtype: int64

**Insight:** Production volume climbs steeply in recent decades — confirming a
dataset skew toward modern releases, which affects any time-series interpretation.

### Q6 — Runtime outliers (IQR method)

In [7]:
q1, q3 = movies.runtime.quantile([0.25, 0.75])
iqr    = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = movies[(movies.runtime < low) | (movies.runtime > high)]
print(f"IQR bounds: {low:.1f} – {high:.1f} minutes | {len(outliers)} outliers")
outliers[["title", "runtime"]].sort_values("runtime")

IQR bounds: 58.5 – 166.5 minutes | 60 outliers


,title,runtime
2404,The Guardians of the Galaxy Holiday Special,45
1641,The Garden of Words,46
2370,Dune: Part Two,167
583,Gangs of New York,168
2154,It Chapter Two,169
1595,Interstellar,169
118,Pirates of the Caribbean: At World's End,169
1306,The Hobbit: An Unexpected Journey,169
344,Saving Private Ryan,169
50,Scarface,170


**Insight:** Runtime outliers are almost all extreme-length epics (>180 min) —
safe to keep, since they are real films, not data errors.

### Q7 — Average popularity by primary genre

In [8]:
# Reload the CSV to preserve the original "first genre listed" order
mg_ordered = pd.read_csv("data/movie_genres.csv")
primary = mg_ordered.groupby("movie_id").first().reset_index()
primary = primary.merge(genres, on="genre_id").merge(movies[["movie_id", "popularity"]], on="movie_id")
avg_pop = primary.groupby("genre_name").popularity.mean().sort_values(ascending=False)
avg_pop.head(15)

genre_name
Science Fiction    21.261836
Music              18.324340
Animation          16.811212
Family             16.710635
Action             16.385188
War                15.470317
Horror             14.851835
Adventure          14.785510
Fantasy            13.022920
Mystery            12.427131
Western            12.322680
Thriller           11.294211
Drama              11.278534
Crime              10.739498
History            10.414567
Name: popularity, dtype: float64

**Insight:** Animation, Adventure and Sci-Fi tend to have the highest average
popularity scores, consistent with their broad family/global appeal.

### Q8 — Top 10 prolific actors + their avg rating

In [9]:
actor_agg = (
    cast.merge(movies[["movie_id", "vote_average"]], on="movie_id")
        .groupby("actor_name")
        .agg(movie_count=("movie_id", "nunique"),
             avg_rating=("vote_average", "mean"))
        .sort_values("movie_count", ascending=False)
        .head(10)
)
actor_agg

,movie_count,avg_rating
actor_name,,
Samuel L. Jackson,39,6.938692
Brad Pitt,38,7.257289
Robert De Niro,38,7.228316
Johnny Depp,37,6.856973
Tom Hanks,33,7.289758
Scarlett Johansson,32,7.056344
Mark Wahlberg,32,6.630063
Willem Dafoe,32,7.114781
Morgan Freeman,31,6.963742


**Insight:** The most prolific actors average solid mid-to-high ratings —
indicating frequent casting correlates with critically accepted films.

### Q9 — Duplicate titles (possible remakes)

In [10]:
dups = movies[movies.duplicated("title", keep=False)].sort_values(["title", "release_year"])
dups[["title", "release_year"]].head(30)

,title,release_year
140,A Nightmare on Elm Street,1984
1164,A Nightmare on Elm Street,2010
323,Aladdin,1992
2068,Aladdin,2019
1042,Alice in Wonderland,1951
1048,Alice in Wonderland,2010
910,Beauty and the Beast,1991
1858,Beauty and the Beast,2017
679,Carrie,1976
1558,Carrie,2013


**Insight:** All repeated titles have different release years — they are remakes
or franchise entries, not duplicate rows. No further de-duplication needed.

### Q10 — Popularity vs vote_count anomalies

In [11]:
p_hi, p_lo = movies.popularity.quantile([0.75, 0.25])
v_hi, v_lo = movies.vote_count.quantile([0.75, 0.25])

high_pop_low_votes = movies[(movies.popularity > p_hi) & (movies.vote_count < v_lo)]
low_pop_high_votes = movies[(movies.popularity < p_lo) & (movies.vote_count > v_hi)]

print(f"High popularity / low votes: {len(high_pop_low_votes)}")
print(f"Low popularity / high votes: {len(low_pop_high_votes)}")
high_pop_low_votes[["title", "popularity", "vote_count"]].head(10)

High popularity / low votes: 35
Low popularity / high votes: 33


,title,popularity,vote_count
316,Lolita,27.5391,2271
983,Malena,23.7035,2753
1030,Shaolin Soccer,19.8665,2605
1270,I Spit on Your Grave,18.1091,2626
1420,A Serbian Film,18.9190,2242
2065,Terrifier,15.6625,3021
2253,Kraven the Hunter,26.1998,2419
2255,Ballerina,23.5962,2836
2267,Lilo & Stitch,33.2161,2228
2288,Final Destination Bloodlines,34.1109,3001


**Insight:** The high-popularity / low-vote group is dominated by very recent releases —
they're trending on TMDB but haven't accumulated votes yet. This is expected
recency bias, not data corruption.